# Tiny Story GPT: training runs on Colab

Three runs, each changing **one** thing from the baseline (ctx256, 0.84M parameters, 4,500 stories, constant learning rate). Baseline full validation loss: **0.8410**.

| Run | Change | Name |
|---|---|---|
| A | cosine learning rate schedule with warmup | `runA_cosine` |
| B | 11× training data (49,500 stories), constant learning rate | `runB_data50k` |
| C | B + bigger model (192-dim, 6 heads, 6 layers), cosine schedule | `runC_bigger` |
| C, re-run | C with a constant learning rate, so it differs from B only in size | `runC_bigger_constant` |

Results: A 0.8743, B 0.8108, C 0.7034, C re-run 0.7027. Cosine hurt the small model and made no difference to the big one.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells in order.
Checkpoints are saved straight to Google Drive, so a disconnect only loses the run in progress.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("CUDA available:", torch.cuda.is_available(), "| torch", torch.__version__)

## 2. Connect Google Drive
Checkpoints and results are kept in `MyDrive/tiny-story-gpt/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/tiny-story-gpt"
!mkdir -p {DRIVE}/checkpoints {DRIVE}/results

## 3. Get the code
`checkpoints/` is a link into Drive, so every saved model lands in Drive directly.

In [ ]:
%cd /content
!rm -rf tiny-story-gpt
!git clone -q https://github.com/spandey1702/tiny-story-gpt.git
%cd /content/tiny-story-gpt
!mkdir -p data
!ln -s {DRIVE}/checkpoints checkpoints
!git log --oneline -1

## 4. Get the data
Streams the first ~48 MB of TinyStoriesV2-GPT4 (about 58k stories) instead of the full 2.2 GB file.
`prepare_data.py` rebuilds the exact same `train.txt` / `valid.txt` as on the Mac, plus `train_49500.txt`.
A `curl: (23) Failure writing output` message is expected: `head` stops reading once it has enough.

In [ ]:
!curl -sL https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt | head -c 48000000 > data/tinystories_raw.txt
!ls -lh data/tinystories_raw.txt
!python src/prepare_data.py --source data/tinystories_raw.txt --extra-stories 45000
!wc -c data/*.txt

## 5. Run A: cosine learning rate schedule
Same model and data as the baseline; only the schedule changes.

In [ ]:
!python src/train.py --steps 10000 --ctx 256 --lr-schedule cosine --name runA_cosine 2>&1 | tee results/train_runA_cosine.log
!cp results/runA_cosine.json results/train_runA_cosine.log {DRIVE}/results/

## 6. Run B: + 10× data
The baseline settings (constant learning rate), trained on 49,500 stories instead of 4,500.

In [ ]:
!python src/train.py --steps 10000 --ctx 256 --train-file data/train_49500.txt --name runB_data50k 2>&1 | tee results/train_runB_data50k.log
!cp results/runB_data50k.json results/train_runB_data50k.log {DRIVE}/results/

## 7. Run C: + bigger model
A ~3× larger model (192-dim, 6 heads, 6 layers) on Run B's data, with the cosine schedule.

In [ ]:
!python src/train.py --steps 10000 --ctx 256 --lr-schedule cosine --train-file data/train_49500.txt --n-embd 192 --n-head 6 --n-layer 6 --name runC_bigger 2>&1 | tee results/train_runC_bigger.log
!cp results/runC_bigger.json results/train_runC_bigger.log {DRIVE}/results/

## 7b. Run C re-run: constant learning rate
The same big model without the cosine schedule, so it changes only the model size compared with Run B.

In [ ]:
!python src/train.py --steps 10000 --ctx 256 --train-file data/train_49500.txt --n-embd 192 --n-head 6 --n-layer 6 --name runC_bigger_constant 2>&1 | tee results/train_runC_bigger_constant.log
!cp results/runC_bigger_constant.json results/train_runC_bigger_constant.log {DRIVE}/results/

## 8. Scoreboard
Full validation loss on the same 500 stories for every run (baseline 0.8410).

In [ ]:
import json

BASELINE = 0.8410
print(f"{'run':22} {'params':>10} {'minutes':>8} {'full val loss':>14} {'vs baseline':>12}")
print(f"{'baseline':22} {842566:>10,} {'(Mac)':>8} {BASELINE:>14.4f} {'':>12}")
for run in ["runA_cosine", "runB_data50k", "runC_bigger", "runC_bigger_constant"]:
    try:
        r = json.load(open(f"results/{run}.json"))
    except FileNotFoundError:
        print(f"{run:22} not run yet")
        continue
    loss = r["full_val_loss"]
    change = (loss - BASELINE) / BASELINE * 100
    print(f"{run:22} {r['parameters']:>10,} {r['train_seconds'] / 60:>8.1f} {loss:>14.4f} {change:>+11.1f}%")

## 9. Download
The checkpoints are already in Drive (`MyDrive/tiny-story-gpt/checkpoints`). This downloads the results and logs as a zip.

In [ ]:
!cd results && zip -q /content/colab_results.zip run*.json train_run*.log
from google.colab import files
files.download("/content/colab_results.zip")